# 🛍️ Analyse Comportementale Clientèle Retail
## Notebook d'Exploration et Préparation des Données (EDA)

**Atelier ML — GI2 | 2025-2026**

Ce notebook couvre :
1. Chargement et aperçu du dataset
2. Analyse des valeurs manquantes
3. Distribution des features numériques
4. Détection des outliers
5. Matrice de corrélation
6. Équilibre des classes (Churn)
7. ACP — Analyse en Composantes Principales
8. Visualisation des segments RFM

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

from utils import (
    plot_missing_values, plot_distributions, plot_correlation_heatmap,
    plot_class_balance, plot_boxplots, plot_pca_variance, plot_pca_2d
)

plt.style.use('dark_background')
print('✅ Imports OK')

## 1. Chargement du Dataset

In [ ]:
df = pd.read_csv('../data/raw/retail_customers.csv', low_memory=False)
print(f'Shape : {df.shape}')
df.head(3)

In [ ]:
print('=== INFO GÉNÉRALES ===')
print(f'Lignes       : {df.shape[0]:,}')
print(f'Colonnes     : {df.shape[1]}')
print(f'\nTypes :')
print(df.dtypes.value_counts())
print(f'\nTarget Churn :')
print(df['Churn'].value_counts())
print(f'\nTaux de churn : {df["Churn"].mean()*100:.1f}%')

## 2. Valeurs Manquantes

In [ ]:
plot_missing_values(df)

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
missing_df[missing_df['Count'] > 0].sort_values('Percentage', ascending=False)

## 3. Distribution des Features Numériques

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.drop(['CustomerID', 'Churn'], errors='ignore').tolist()
print(f'{len(num_cols)} features numériques')
plot_distributions(df, num_cols[:16], n_cols=4)

## 4. Détection des Outliers (Boxplots)

In [ ]:
outlier_cols = ['MonetaryTotal', 'TotalQuantity', 'SupportTicketsCount',
                'SatisfactionScore', 'ReturnRatio', 'Frequency', 'Recency']
outlier_cols = [c for c in outlier_cols if c in df.columns]
plot_boxplots(df, outlier_cols, n_cols=4)

In [ ]:
# Valeurs aberrantes identifiées
print('=== VALEURS ABERRANTES IDENTIFIÉES ===')
print()
if 'SupportTicketsCount' in df.columns:
    bad_support = df['SupportTicketsCount'].isin([-1, 999]).sum()
    print(f'SupportTicketsCount (-1 ou 999) : {bad_support} lignes')
if 'SatisfactionScore' in df.columns:
    bad_sat = (~df['SatisfactionScore'].between(0, 5)).sum()
    print(f'SatisfactionScore hors [0-5]    : {bad_sat} lignes')
if 'MonetaryTotal' in df.columns:
    neg_monetary = (df['MonetaryTotal'] < 0).sum()
    print(f'MonetaryTotal négatif           : {neg_monetary} lignes')

## 5. Matrice de Corrélation

In [ ]:
num_df = df.select_dtypes(include=np.number).drop(['CustomerID'], errors='ignore')
high_corr_pairs = plot_correlation_heatmap(num_df, threshold=0.8)

## 6. Distribution des Variables Catégorielles & Churn

In [ ]:
cat_targets = ['Churn', 'RFMSegment', 'CustomerType', 'SpendingCategory',
               'Region', 'AccountStatus', 'ChurnRiskCategory']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, col in enumerate(cat_targets):
    if col not in df.columns or i >= len(axes):
        continue
    vc = df[col].value_counts()
    axes[i].bar(vc.index.astype(str), vc.values,
                color=sns.color_palette('Set2', len(vc)))
    axes[i].set_title(col, fontsize=11)
    axes[i].tick_params(axis='x', rotation=30)
    for j, v in enumerate(vc.values):
        axes[i].text(j, v + 2, str(v), ha='center', fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution des Variables Catégorielles', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/cat_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Analyse Churn par Feature

In [ ]:
churn_analysis_cols = ['Recency', 'Frequency', 'MonetaryTotal',
                       'SatisfactionScore', 'CustomerTenureDays', 'Age']
churn_analysis_cols = [c for c in churn_analysis_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(churn_analysis_cols):
    churned     = df[df['Churn'] == 1][col].dropna()
    not_churned = df[df['Churn'] == 0][col].dropna()
    axes[i].hist(not_churned, bins=30, alpha=0.6, label='Fidèle (0)', color='#3b82f6')
    axes[i].hist(churned,     bins=30, alpha=0.6, label='Churné (1)', color='#ef4444')
    axes[i].set_title(col); axes[i].legend(fontsize=8)

plt.suptitle('Distribution par Churn', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/churn_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. ACP — Analyse en Composantes Principales

In [ ]:
# Préparer les données pour l'ACP (numériques uniquement)
df_num = df.select_dtypes(include=np.number).drop(
    ['CustomerID', 'Churn'], errors='ignore'
).dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_num)

# ACP complète
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

plot_pca_variance(pca_full)
plt.show()

In [ ]:
# Projection 2D colorée par Churn
pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_scaled)

churn_labels = df.loc[df_num.index, 'Churn'].values

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                     c=churn_labels, cmap='RdYlGn_r',
                     alpha=0.5, s=12, edgecolors='none')
plt.colorbar(scatter, label='Churn (1=parti)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('ACP 2D — Coloré par Churn')
plt.tight_layout()
plt.savefig('../reports/pca_churn_2d.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Variance PC1 : {pca_2d.explained_variance_ratio_[0]*100:.1f}%')
print(f'Variance PC2 : {pca_2d.explained_variance_ratio_[1]*100:.1f}%')

## 9. Résumé Statistique Final

### Conclusions de l'EDA
- **Valeurs manquantes** : Age (~30%), RegistrationDate (formats inconsistants)
- **Outliers** : SupportTicketsCount (-1, 999), SatisfactionScore (-1, 99), MonetaryTotal (<0)
- **Multicolinéarité** : MonetaryMin/Max fortement corrélées à MonetaryTotal
- **Déséquilibre classes** : Churn ~minoritaire → SMOTE nécessaire
- **Feature Engineering** : Parsing dates, ratios métier, IP engineering

→ Lancer `src/preprocessing.py` puis `src/train_model.py`

In [ ]:
print('=== RÉSUMÉ STATISTIQUE ===')
print(df.describe(include='all').T[['count','mean','std','min','max']].to_string())